# Modélisation

### Import des modules

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import bentoml

from sklearn.model_selection import KFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.multioutput import MultiOutputRegressor

# Modèles
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor

# Métriques
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Chargement du dataset préparé

In [ ]:
# Prépare le chemin du dataset "clean" préparé lors de l'eda
DATA_DIR = Path.home() / "Desktop" / "Data Engineer" / "Projet 06"
EDA_NAME = "df_eda_clean.csv"
PATH_EDA = DATA_DIR / EDA_NAME

# Targets
TARGET_ENERGY = "SiteEnergyUse(kBtu)"
TARGET_GHG = "TotalGHGEmissions"
TARGETS = [TARGET_ENERGY, TARGET_GHG]

# Random_State
RANDOM_STATE = 42
N_SPLITS = 5

# Vérifie que le fichier existe
if not PATH_EDA.exists():
    raise FileNotFoundError(f"Fichier introuvable : {PATH_EDA}")

# Import Dataset
df = pd.read_csv(PATH_EDA)

# Vérifie s'il manque des targets
missing_targets = [c for c in TARGETS if c not in df.columns]
if missing_targets:
    raise ValueError(f"Targets manquantes dans le dataset : {missing_targets}")
    
print("Dataset chargé avec succès")
print(f"Dimensions : {df.shape}")
print(df.head())

### Feature Engineering

In [ ]:
# Séparation des variables explicatives (features) et des variables cibles (targets)
X = df.drop(columns=TARGETS)
y = df[TARGETS].copy()

if X.empty:
    raise ValueError("Aucune feature disponible après suppression des targets.")

print("Séparation features / targets effectuée")
print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")

### Préparation des features pour la modélisation

In [ ]:
# Features
REQUIRED_FE_COLS = [
    "PropertyGFATotal",
    "PropertyGFABuilding(s)",
    "PropertyGFAParking",
    "NumberofBuildings",
    "NumberofFloors",
    "YearBuilt",
    "LargestPropertyUseTypeGFA",
    "Latitude",
    "Longitude"
]

missing_cols = [c for c in REQUIRED_FE_COLS if c not in X.columns]
if missing_cols:
    raise KeyError(f"Colonnes manquantes pour le feature engineering : {missing_cols}")

# Constantes
DATASET_YEAR = 2016
MULTI_USE_THRESHOLD = 0.9
FLOORS_BINS = [0, 2, 5, 10, 20, np.inf]
FLOORS_LABELS = ["1-2", "3-5", "6-10", "11-20", "20+"]

GFA_TOTAL = X["PropertyGFATotal"].replace(0, np.nan)
NB_BUILDINGS = X["NumberofBuildings"].replace(0, np.nan)

# Ratios de surfaces
X["GFA_Building_Ratio"] = X["PropertyGFABuilding(s)"] / GFA_TOTAL
X["GFA_Parking_Ratio"] = X["PropertyGFAParking"] / GFA_TOTAL
# Réduit l'effet des grands bâtiments
X["Log_PropertyGFATotal"] = np.log1p(X["PropertyGFATotal"])

# Âge du bâtiment 
X["Building_Age"] = DATASET_YEAR - X["YearBuilt"]
# Décennie de construction
X["YearBuilt_Decade"] = (X["YearBuilt"] // 10 * 10).astype("Int64")

# Ratio étages / bâtiments
X["Floors_per_Building"] = X["NumberofFloors"] / NB_BUILDINGS
# Classes d'étages
X["Floors_Class"] = pd.cut(
    X["NumberofFloors"],
    bins=FLOORS_BINS,
    labels=FLOORS_LABELS
)

# Usage principal
X["LargestUse_GFA_Ratio"] = X["LargestPropertyUseTypeGFA"] / GFA_TOTAL
X["Is_MultiUse"] = (X["LargestUse_GFA_Ratio"] < MULTI_USE_THRESHOLD).astype("Int64")


# Nettoyage des valeurs infinies créées par les ratios
X.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Feature engineering terminé")
print(f"Nombre de features : {X.shape[1]}")

In [ ]:
NUM_FEATURES = [
    # taille
    "PropertyGFATotal",
    "Log_PropertyGFATotal",
    "GFA_Building_Ratio",
    "GFA_Parking_Ratio",

    # temporalité
    "Building_Age",

    # structure
    "NumberofBuildings",
    "NumberofFloors",
    "Floors_per_Building",

    # usage
    "LargestUse_GFA_Ratio",

    # localisation
    "Latitude",
    "Longitude"
]

CAT_FEATURES = [
    "PrimaryPropertyType",
    "LargestPropertyUseType",
    "Neighborhood",
    "Floors_Class",
    "YearBuilt_Decade",
]

BIN_FEATURES = [
    "Is_MultiUse"
]

# Vérification de la présence des features
ALL_FEATURES = NUM_FEATURES + CAT_FEATURES + BIN_FEATURES
missing_features = [c for c in ALL_FEATURES if c not in X.columns]

if missing_features:
    raise KeyError(f"Features absentes dans X : {missing_features}")

In [ ]:
ALL_FEATURES = NUM_FEATURES + CAT_FEATURES + BIN_FEATURES

# Debug, colonnes ignorées par le modèle
extra_in_X = sorted(set(X.columns) - set(ALL_FEATURES))
if extra_in_X:
    print("Colonnes ignorées (non utilisées par le modèle) :", extra_in_X)

# On restreint X aux features utilisées par le modèle
X = X[ALL_FEATURES].copy()

print(f"X restreint aux {len(ALL_FEATURES)} features finales.")
print("Dimensions finales de X :", X.shape)

In [ ]:
# ======================================
# Préprocessing - LinearRegression, SVR
# ======================================

# Preprocessing des variables numériques
numeric_transformer_linear = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocessing des variables catégorielles
categorical_transformer_linear = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        max_categories=15,
        sparse_output=False
    ))
])

# Preprocessing des variables binaires
binary_transformer_linear = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

# Preprocessor global pour les modèles linéaires
preprocessor_linear = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_linear, NUM_FEATURES),
        ("cat", categorical_transformer_linear, CAT_FEATURES),
        ("bin", binary_transformer_linear, BIN_FEATURES)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# Modèles de la famille linéaire
models_linear = {
    "LinearRegression": LinearRegression(),
    "SVR": MultiOutputRegressor(SVR()),
}

In [ ]:
# =============================
# Préprocessing - RandomForest
# =============================

# Preprocessing des variables numériques
numeric_transformer_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
])

# Preprocessing des variables catégorielles
categorical_transformer_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        max_categories=15,
        sparse_output=False
    ))
])

# Preprocessing des variables binaires
binary_transformer_tree = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

# Preprocessor global pour RandomForest
preprocessor_tree = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer_tree, NUM_FEATURES),
        ("cat", categorical_transformer_tree, CAT_FEATURES),
        ("bin", binary_transformer_tree, BIN_FEATURES)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# Modèles RandomForest
models_tree = {
    "RandomForest": RandomForestRegressor(
        n_estimators=300,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

In [ ]:
# ======================
# Préprocessing - Dummy
# ======================
preprocessor_dummy = preprocessor_tree

models_dummy = {
    "Dummy_Mean": DummyRegressor(strategy="mean"),
}

In [ ]:
# Protocole d'évaluation finale des modèles

cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "r2": "r2",
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error",
    "rmse": "neg_root_mean_squared_error"
}

In [ ]:
# Fonction d'évaluation des modèles
def evaluate_models(X, y, models, preprocessor, cv, scoring):
    results = []

    for name, model in models.items():
        pipeline = Pipeline(steps=[
            ("preprocessing", preprocessor),
            ("model", model)
        ])

        cv_results = cross_validate(
            estimator=pipeline, 
            X=X,
            y=y,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            return_train_score=False
        )

        results.append({
            "Model": name,
            "R2_mean": cv_results["test_r2"].mean(),
            "R2_std": cv_results["test_r2"].std(),
            "MAE_mean": -cv_results["test_mae"].mean(),
            "MSE_mean": -cv_results["test_mse"].mean(),
            "RMSE_mean": -cv_results["test_rmse"].mean()
        })

    return pd.DataFrame(results).sort_values("R2_mean", ascending=False).reset_index(drop=True)

### Comparaison des modèles supervisés

In [ ]:
# Evaluation globale des modèles

# Evaluation pour les modèles linéaires
results_linear = evaluate_models(
    X=X,
    y=y,
    models=models_linear,
    preprocessor=preprocessor_linear,
    cv=cv,
    scoring=scoring
)

# Evaluation pour RandomForest
results_tree = evaluate_models(
    X=X,
    y=y,
    models=models_tree,
    preprocessor=preprocessor_tree,
    cv=cv,
    scoring=scoring
)


results_dummy = evaluate_models(
    X=X,
    y=y,
    models=models_dummy,
    preprocessor=preprocessor_dummy,
    cv=cv,
    scoring=scoring
)


print("Résumé des performances des modèles : ")
#print(results)

# Agrégation finale
results_all = (
    pd.concat([results_linear, results_tree, results_dummy])
      .sort_values("MAE_mean")
      .reset_index(drop=True)
)

results_all

### Optimisation et interprétation du modèle

In [ ]:
# Optimisation du meilleur modèle

param_grid = {
    "model__n_estimators": [400, 600, 800],
    "model__max_depth": [None, 20, 40, 60],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 5],
    "model__max_features": ["sqrt"]
}

pipe_rf = Pipeline(steps=[
    ("preprocessing", preprocessor_tree),
    ("model", RandomForestRegressor(
        random_state=RANDOM_STATE, 
        n_jobs=-1
    ))
])

gs = GridSearchCV(
    estimator=pipe_rf,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

gs.fit(X, y)

print("GridSearch terminé")
print(f"Best MAE (CV mean) : {-gs.best_score_:.2f}")
print("Best parameters :")
for k, v in gs.best_params_.items():
    print(f"  - {k}: {v}")

In [ ]:
# Extraction des meilleurs hyperparamètres
best_rf_params = {
    k.replace("model__", ""): v
    for k, v in gs.best_params_.items()
}

rf_final = RandomForestRegressor(
    **best_rf_params,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor_tree),
    ("model", rf_final)
])

# Entraînement sur l'ensemble du dataset
final_pipeline.fit(X, y)

print("Modèle entraîné sur l'ensemble du dataset")

# Analyse de l'importance des features du modèle

In [ ]:
# Récupération des éléments du pipeline final
preproc = final_pipeline.named_steps["preprocessing"]
model = final_pipeline.named_steps["model"]

# Nom et Importances des features
feature_names = preproc.get_feature_names_out()
importances = model.feature_importances_

if len(feature_names) != len(importances):
    raise ValueError("Incohérence entre le nombre de features et les importances")

fi_df = (
    pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    })
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("Top 15 des features les plus importantes :")
print(fi_df.head(15))

In [ ]:
TOP_N = 15

if fi_df.empty:
    raise ValueError("fi_df est vide : impossible d'afficher les feature importances.")

top_features = fi_df.head(TOP_N)

plt.figure(figsize=(10, 6))
plt.barh(
    top_features["feature"][::-1],
    top_features["importance"][::-1]
)
plt.title(f"Top {TOP_N} features les plus importantes du modèle final")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

### Sauvegarde BentoML

In [ ]:
# Sauvegarde du modèle final avec BentoML
MODEL_NAME = "seattle_energy_model"

bentoml.sklearn.save_model(
    name=MODEL_NAME,
    model=final_pipeline,
    metadata={
        "targets": TARGETS,
        "problem_type": "multi_output_regression",
        "city": "Seattle",
        "model_type": "RandomForestRegressor",
        "features_count": X.shape[1],
        "dataset_rows": X.shape[0]
    }
)

print(f"Modèle '{MODEL_NAME}' sauvegardé avec succès dans BentoML")